# РК1 · Автосервис · Qwen2.5-3B + QLoRA

**Гулямов Рахим.** Подготовлено с помощью ИИ. Все экспериментальные результаты получаются запуском.

1. **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить.**
2. **Среда выполнения → Выполнить всё.**
3. Дождаться скачивания `autoservice_results.zip`; дополнительно сохранить выполненный `.ipynb`.

Код и данные уже внутри notebook, подключать Drive и вводить API-ключ не нужно.
Ориентир времени — 40–90 минут на T4 после выделения GPU, но это не измеренный результат.
Если ячейка красная — не продолжать вслепую: сохранить ошибку и исправить причину.


## 1. Распаковка исходников и проверка GPU

Повторная распаковка поверх существующего запуска запрещена, чтобы случайно не подменить конфигурацию. Для продолжения запуска пропустите эту ячейку.

In [ ]:
import base64, io, zipfile, pathlib, os, sys, subprocess, datetime
PROJECT = pathlib.Path("/content/autoservice_sft")
if PROJECT.exists():
    raise RuntimeError("Проект уже распакован. Для продолжения запускайте с нужного этапа, пропустив эту ячейку.")
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall("/content")
os.chdir(PROJECT)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
(PROJECT / "outputs").mkdir(exist_ok=True)
(PROJECT / "outputs/run_started_utc.txt").write_text(datetime.datetime.now(datetime.timezone.utc).isoformat())

def run(args, log_name):
    log = PROJECT / "outputs" / log_name
    print("Запуск:", " ".join(args), flush=True)
    with log.open("w", encoding="utf-8") as out:
        proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True)
            out.write(line); out.flush()
        status = proc.wait()
    if status:
        raise RuntimeError(f"Этап завершился с кодом {status}. Лог: {log}. Пришлите последние строки ошибки.")

run([sys.executable, "-c", "import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ'); assert torch.cuda.is_available(), 'Выберите T4 GPU'"], "00_gpu.log")


## 2. Установка и CPU sanity check

Устанавливаются библиотеки из PyPI. PyTorch остаётся из GPU-среды Colab. Восемь тестов автоматической проверки выполняются до работы с моделью.

In [ ]:
run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-colab.txt"], "01_install.log")
run([sys.executable, "build_data.py"], "02_data.log")
run([sys.executable, "checks.py"], "03_checks.log")
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(PROJECT / "outputs/pip-freeze.txt").write_text(freeze)
print("Установка и проверки завершены.")


## 3. Бейзлайн

Скачивание 3B-модели может занять несколько минут. Сохраняются все 24 ответа, первые два печатаются. Веса и данные обучения пока не использованы для генерации.

In [ ]:
run([sys.executable, "experiment.py", "before"], "04_before.log")


## 4. Первое дообучение

Проверяется answer-only loss и градиенты, затем выполняются 50 шагов QLoRA. Train loss и dev loss сохраняются; адаптер записывается в safetensors.

In [ ]:
run([sys.executable, "experiment.py", "train"], "05_train.log")


## 5. Проверка сохранённого адаптера

Отдельный процесс заново загружает модель и файлы адаптера. Те же 24 вопроса, инструкция и decoding.

In [ ]:
run([sys.executable, "experiment.py", "after"], "06_after.log")


## 6. Отдельная модель-судья

После освобождения памяти загружается Qwen2.5-7B-Instruct. Шесть заранее выбранных пар оцениваются в двух порядках A/B. Скачивание этой модели больше, чем 3B; это ожидаемо. API-ключ и оплата не используются.

In [ ]:
run([sys.executable, "experiment.py", "judge"], "07_judge.log")


## 7. Метрики, графики и фактический отчёт

Отчёт собирается из сохранённых ответов, логов и оценок. Невалидные оценки судьи не подменяются числами.

In [ ]:
run([sys.executable, "summarize.py"], "08_summary.log")
from IPython.display import display, Markdown, Image
display(Markdown((PROJECT / "outputs/metrics.csv").read_text()))
display(Image(filename=str(PROJECT / "outputs/comparison.png")))
display(Image(filename=str(PROJECT / "outputs/loss.png")))
print("Отчёт:", PROJECT / "REPORT.md")


## 8. Скачать пакет для сдачи

Архив содержит код, данные, REPORT.md, метрики, графики, логи и адаптер. Перед LMS нужно опубликовать репозиторий и доступную преподавателю ссылку на веса. Отчёт прочитать и проверить; ссылки в SUBMISSION.md заполнить.

In [ ]:
import shutil
from google.colab import files
required = ["REPORT.md", "outputs/before.jsonl", "outputs/after.jsonl", "outputs/metrics.csv", "outputs/adapter/adapter_model.safetensors"]
for item in required:
    assert (PROJECT / item).exists(), f"Нет обязательного артефакта: {item}"
result = shutil.make_archive("/content/autoservice_results", "zip", root_dir="/content", base_dir="autoservice_sft")
print("ГОТОВО. Сохраните архив и выполненный notebook.")
files.download(result)


## Аварийное сохранение (вручную при необходимости)

Если один из этапов упал, можно отдельно выполнить эту ячейку и скачать то, что уже получено. **Это неполный результат**; архив не считается готовой работой.

In [ ]:
# Выполняется только если пользователь изменит флаг на True.
SAVE_PARTIAL = False
if SAVE_PARTIAL:
    import shutil
    from google.colab import files
    files.download(shutil.make_archive("/content/autoservice_PARTIAL", "zip", root_dir="/content", base_dir="autoservice_sft"))
